# Lecture: From Pixels to Meters — Detection, Tracking, Homography (Sports Pipeline)

Read this before opening notebook.ipynb. Each vocabulary term below gets its own section
(marked §) with a picture or worked numbers — skim the index, study the ones that feel fuzzy.

## Learning goals

- Explain the pipeline detect → track → map-to-meters in one paragraph (§8).
- Explain what detection, centroid tracking with coasting, and homography each contribute (§4–§6).
- Read one centroid's journey from pixels to meters in the §8 trace.


## The big idea

Imagine a sports analyst with three jobs. First, each video frame, circle every player with a crayon (**detection**: blobs of team colors become boxes — touching pixels are grouped by **connected components**, and each blob's **centroid** becomes the player's position that frame). Second, give each circled player a name tag that sticks across frames (**tracking**: this frame's circle near last frame's "Mia" is still Mia if it falls inside the **GATE** distance; if she hides behind someone, keep her tag alive by guessing — **coasting** — and watch out for **ID switches** where one player collects two tags). Third, translate TV pixels into real meters (**homography**: a 3×3 matrix un-tilts the camera view into a top-down map, solved from 4 known corners by the **DLT** recipe, checked by **reprojection error**, and summarized as a **heatmap** of where everyone spent time).


> Lab stand-in, stated upfront: the curriculum names YOLO, but this lab detects with **color-threshold + connected-component blobs** (fast, CPU-only, works on synthetic kits). A real YOLO learns boxes from pixels and survives lighting/kit changes; color masks break the moment colors overlap. The tracking/homography/heatmap machinery downstream is identical either way.

In [ ]:
import numpy as np  # bring in numpy for grid points and matrix math
import matplotlib.pyplot as plt  # bring in pyplot for the before/after panels
np.random.seed(9)  # fix the seed so the demo is identical every run
gx, gy = np.meshgrid(np.linspace(0, 4, 5), np.linspace(0, 3, 4))  # 5x4 top-down pitch grid
pts = np.stack([gx.ravel(), gy.ravel()], axis=1)  # flatten the grid to (20, 2) points
H = np.array([[1.0, 0.3, 1.0],  # homography row 0: tilt plus shift in x
              [0.1, 1.0, 0.5],  # homography row 1: tilt plus shift in y
              [0.05, 0.02, 1.0]])  # homography row 2: the perspective squeeze
hom = np.concatenate([pts, np.ones((len(pts), 1))], axis=1)  # lift points to homogeneous coords
warp = (H @ hom.T).T  # multiply every point by the 3x3 matrix
cam = warp[:, :2] / warp[:, 2:3]  # divide by w: the perspective step giving camera pixels
back = np.linalg.inv(H) @ np.concatenate([cam, np.ones((len(cam), 1))], axis=1).T  # invert the warp
back = (back.T[:, :2] / back.T[:, 2:3])  # divide by w to return to meters
print('round-trip max err:', round(float(np.abs(back - pts).max()), 10))  # ~0.0 proves the map inverts
print('meaning: 4 known corners pin down H, then every pixel maps to meters')  # the intuition line
fig, ax = plt.subplots(1, 2, figsize=(8, 4))  # make two panels: map vs camera
ax[0].scatter(pts[:, 0], pts[:, 1], c='green')  # draw the true top-down grid
ax[0].set_title('pitch map (meters, top-down)')  # title the left panel
ax[0].set_aspect('equal')  # keep meters square on the left
ax[1].scatter(cam[:, 0], cam[:, 1], c='blue')  # draw the tilted camera view
ax[1].set_title('camera view (pixels, tilted)')  # title the right panel
ax[1].set_aspect('equal')  # keep shapes honest on the right
fig.suptitle('Homography un-tilts the camera view back to meters')  # overall takeaway
fig.tight_layout()  # tidy spacing between panels
plt.show()  # render the figure


## Vocabulary, one section each

Quick map — jump to whatever feels fuzzy. §4 circles players, §5 names them across frames, §6 converts to meters, §7 summarizes.

| Term | One-liner | Section |
|---|---|---|
| Detection | Circling players in one frame (the lab: color masks + blobs become boxes) | §4 |
| Connected components | Grouping touching foreground pixels into one blob (= one player) | §4 |
| Centroid | A blob's center point, used as the player's position that frame | §4 |
| Tracker / gating | Linking frames: a detection within GATE pixels of a track joins it | §5 |
| Coasting | Keeping a hidden player's tag alive at its last spot for a few frames | §5 |
| ID switch | Bug where one player collects two tags (lab: 11 IDs for 9 entities) | §5 |
| Homography | A 3×3 matrix mapping tilted camera pixels to top-down pitch meters | §6 |
| DLT (direct linear transform) | The equation-solving recipe finding the homography from 4 corners | §6 |
| Reprojection error | Leftover meters of error on known corners (~0 means the map fits) | §6 |
| Heatmap | A 2D histogram of where players spent time on the pitch | §7 |


### §4 Detection, connected components, centroid

**In one sentence: detection thresholds team colors into masks, connected-components groups touching mask pixels into one blob per player, and the centroid (the blob's mean x, mean y) becomes that player's position for the frame.**

Color comes first because it is cheap: "very red" pixels are team A, "very blue" are team B, near-white are the ball — the pitch lines are dimmer gray so they stay out of every mask. Then flood-fill (breadth-first search) claims each touching clump as one component, and specks below a minimum size are thrown away as noise. Watch it: our 8×8 mask holds one 3×3 blob (rows 2–4, cols 3–5) plus one noise pixel — the blob's centroid is (x 4.0, y 3.0) with box x 3–5 / y 2–4, and the lone pixel is ignored exactly like the lab's size filter ignores speckle.


In [ ]:
import numpy as np  # reuse numpy for the mask and centroid math
import matplotlib.pyplot as plt  # reuse pyplot for the mask picture
import matplotlib.patches as patches  # reuse patches to draw the box
from collections import deque  # BFS queue: flood-fill claims touching pixels level by level
np.random.seed(9)  # fix the seed for reproducibility
mask = np.zeros((8, 8), dtype=int)  # empty 8x8 color mask: 1 = team-A pixel
mask[2:5, 3:6] = 1  # paint one 3x3 blob (rows 2-4, cols 3-5): our player
mask[6, 6] = 1  # paint one lone noise pixel far from the blob
seen = np.zeros_like(mask, dtype=bool)  # visited map: never flood the same pixel twice
comps = []  # collected components: each is a list of (y, x) pixels
for y in range(8):  # scan every row for an unclaimed foreground pixel
    for x in range(8):  # scan every column in this row
        if mask[y, x] == 0 or seen[y, x]:  # skip background and already-claimed pixels
            continue  # nothing to seed here
        q = deque([(y, x)])  # seed a new flood from this pixel
        seen[y, x] = True  # mark the seed visited at enqueue time (no double-queueing)
        cur = []  # pixels of the component growing from this seed
        while q:  # drain the frontier level by level
            cy0, cx0 = q.popleft()  # take the oldest frontier pixel
            cur.append((cy0, cx0))  # claim it for this component
            for dy, dx in ((1, 0), (-1, 0), (0, 1), (0, -1)):  # 4-neighbours: up/down/left/right touch
                ny, nx = cy0 + dy, cx0 + dx  # neighbour coordinates
                if 0 <= ny < 8 and 0 <= nx < 8 and mask[ny, nx] == 1 and not seen[ny, nx]:  # inside, foreground, unvisited
                    seen[ny, nx] = True  # mark at enqueue: each pixel queued once
                    q.append((ny, nx))  # frontier grows by one touching pixel
        comps.append(cur)  # flood ended: one full blob collected
comps = [c for c in comps if len(c) >= 2]  # size filter: drop lone-pixel speckle like the lab does
blob = np.array(comps[0])  # our player: the surviving 9-pixel component
ys, xs = blob[:, 0], blob[:, 1]  # split into y and x coordinates
cx, cy = float(xs.mean()), float(ys.mean())  # centroid: mean x, mean y of the blob
x0, x1, y0, y1 = int(xs.min()), int(xs.max()), int(ys.min()), int(ys.max())  # bounding box edges
print('centroid: x =', round(cx, 1), ', y =', round(cy, 1))  # expect (4.0, 3.0): the blob's center
print('box: x', x0, '-', x1, ', y', y0, '-', y1)  # expect x 3-5, y 2-4
print('components found:', len(comps), '+ 1 noise pixel dropped by the size filter')  # 1 kept blob, 1 speck dropped
fig, ax = plt.subplots(figsize=(4, 4))  # make one panel for the mask
ax.imshow(mask, cmap='gray_r')  # draw the mask: white = player pixels
ax.add_patch(patches.Rectangle((x0 - 0.5, y0 - 0.5), x1 - x0 + 1, y1 - y0 + 1, fill=False, edgecolor='red', linewidth=2))  # red box
ax.plot(cx, cy, 'b*', ms=14)  # blue star: the centroid position
ax.set_title('mask -> blob box + centroid star (speck ignored)')  # title the panel
fig.tight_layout()  # tidy spacing around the panel
plt.show()  # render the figure


### §5 Tracker, gating, coasting, ID switches

**In one sentence: the tracker links each frame's centroids to last frame's tracks by nearest distance within GATE pixels, coasts unmatched tracks at their last spot for up to MAX_MISS frames, and suffers an ID switch whenever one player accidentally collects two tags.**

GATE is a trust radius: jumps under 12 px are "same player walking", jumps over it mean "someone new appeared". Coasting is the tracker's patience — a player hidden for 2 frames with MAX_MISS = 3 stays alive and re-links, dying only after 3 straight misses. ID switches are the failure mode: when occlusion splits one trail in two, the counter inflates — the lab creates 11 IDs for 9 true entities. Watch it: Mia is seen at frames 0–1, hidden at 2–3 (coasted, misses 1 then 2), and re-linked at frame 4 — alive throughout, no new tag.


In [ ]:
import numpy as np  # reuse numpy for positions and distances
import matplotlib.pyplot as plt  # reuse pyplot for the tracking timeline
np.random.seed(9)  # fix the seed so the toy clip is identical every run
GATE, MAX_MISS = 12.0, 3  # trust radius in px; patience in missed frames (lab values)
seen = {0: (10., 10.), 1: (12., 11.), 4: (16., 13.), 5: (18., 14.)}  # Mia sightings: hidden at frames 2-3
pos, misses, alive = (10., 10.), 0, True  # track state: starts at the frame-0 sighting
coasted = []  # record which frames were coasted (guessed, not seen)
for t in range(6):  # walk six toy frames
    if t in seen:  # detection exists this frame
        d = abs(seen[t][0] - pos[0]) + abs(seen[t][1] - pos[1])  # L1 jump from last known spot
        link = bool(d <= GATE)  # inside the trust radius means same player
        pos, misses = seen[t], 0  # snap to the sighting, reset patience
        print(f'frame {t}: seen {seen[t]}, jump {d:.1f} -> link={link}')  # narrate the link
        coasted.append(False)  # real observation, not a guess
    else:  # nobody seen: coast
        misses += 1  # one more missed frame
        alive = bool(misses <= MAX_MISS)  # survive only within patience
        print(f'frame {t}: hidden -> coast at {pos}, misses={misses}, alive={alive}')  # narrate the coast
        coasted.append(True)  # guessed point
print('tags used: 1 (no ID switch: the gap never exceeded MAX_MISS)')  # the happy ending
fig, ax = plt.subplots(figsize=(7, 3.5))  # make one panel for the timeline
ts = np.arange(6)  # frame numbers 0..5
xs = np.array([10., 12., 12., 12., 16., 18.])  # x trail: flat while coasting, steps when seen
ax.plot(ts, xs, 'k:', alpha=0.5)  # dotted line: the track's memory
ax.scatter(ts[[0, 1, 4, 5]], xs[[0, 1, 4, 5]], c='green', s=80, label='seen')  # green dots: real sightings
ax.scatter(ts[[2, 3]], xs[[2, 3]], facecolors='none', edgecolors='orange', s=80, label='coasted')  # hollow dots: guesses
ax.set_xticks(ts)  # one tick per frame
ax.set_xlabel('frame')  # label time
ax.set_ylabel('x position (px)')  # label the trail
ax.set_title('Mia drops out at 2-3, coasts, re-links at 4 (GATE=12, MAX_MISS=3)')  # title the story
ax.legend()  # show which dots are real vs guessed
plt.tight_layout()  # tidy spacing around the panel
plt.show()  # render the figure


### §6 Homography, DLT, reprojection error

**In one sentence: a homography is the 3×3 matrix mapping tilted camera pixels to top-down pitch meters, DLT is the recipe solving it from 4 known corner pairs, and reprojection error is the leftover corner mismatch in meters (~0 means the map fits).**

Each corner pair contributes two linear equations (`A h = 0`); stacking all 8 and taking the singular vector for the smallest singular value yields the 9 matrix entries (normalized so `H[2,2] = 1`). Applying `H` means appending a 1 (homogeneous lift), multiplying, then dividing by the third coordinate — that division is the perspective "un-tilt". Watch it: solving from the lab's own pitch-rect corners gives max corner error ~1e-12 m — essentially zero (exact digits vary by machine; the claim is "roundtrip error ≈ 0"), and the frame center (80, 60) lands at (20.0, 15.0) m — the middle of the 40×30 pitch, exactly as symmetry demands.


In [ ]:
import numpy as np  # reuse numpy for the DLT solve and mapping math
import matplotlib.pyplot as plt  # reuse pyplot for the corners plot
np.random.seed(9)  # fix the seed for reproducibility
SRC = np.array([[12., 12.], [148., 12.], [148., 108.], [12., 108.]])  # 4 pitch-rect corners in pixels (lab frame)
DST = np.array([[0., 0.], [40., 0.], [40., 30.], [0., 30.]])  # same corners in meters (40x30 pitch)
A = []  # 2 rows per corner pair: solve A h = 0
for (x, y), (X, Y) in zip(SRC, DST):  # each correspondence adds two equations
    A.append([-x, -y, -1, 0, 0, 0, X * x, X * y, X])  # row for the X equation
    A.append([0, 0, 0, -x, -y, -1, Y * x, Y * y, Y])  # row for the Y equation
_, _, Vt = np.linalg.svd(np.array(A))  # null space = last row of V-transpose
Hm = Vt[-1].reshape(3, 3)  # reshape the 9-vector into 3x3
Hm = Hm / Hm[2, 2]  # normalize so H[2,2] = 1
def apply_H(Hm, pts):  # map Nx2 pixel points to pitch meters
    p = np.asarray(pts, dtype=float)  # ensure a float array
    h = (Hm @ np.concatenate([p, np.ones((len(p), 1))], axis=1).T).T  # homogeneous multiply
    return h[:, :2] / h[:, 2:3]  # divide by w: the perspective step
reproj = float(np.abs(apply_H(Hm, SRC) - DST).max())  # worst corner mismatch in meters
center = apply_H(Hm, [(80., 60.)])[0]  # map the frame center to meters
print(f'max corner (reprojection) error (m): {reproj:.1e}')  # expect 2.0e-12: essentially zero
print('frame center (80, 60) -> meters:', np.round(center, 2))  # expect [20, 15]: middle of the pitch
fig, ax = plt.subplots(1, 2, figsize=(8, 3.5))  # two panels: pixels vs meters
ax[0].scatter(SRC[:, 0], SRC[:, 1], c='blue')  # the 4 corners in pixel space
ax[0].set_title('corners in pixels (tilted frame)')  # title the left panel
ax[0].set_aspect('equal')  # keep shapes honest
ax[1].scatter(DST[:, 0], DST[:, 1], c='green')  # the 4 corners in meter space
ax[1].set_title('corners in meters (top-down pitch)')  # title the right panel
ax[1].set_aspect('equal')  # keep meters square
fig.suptitle('DLT turns 4 known corners into every-pixel meters')  # overall takeaway
fig.tight_layout()  # tidy spacing between panels
plt.show()  # render the figure


### §7 Heatmap — where did everyone spend time

**In one sentence: a heatmap is a 2D histogram counting how many tracked positions fall in each pitch zone, so bright zones mark camping spots and dark zones mark neglected grass.**

It is the cheapest statistic in the pipeline — no motion model, just binning — yet coaches love it: one glance shows whether the left wing was abandoned. Watch it: 44 toy visits split `[12, 5, 7, 20]` across four pitch quarters, so zone 3 (bottom-right) glows with 20 visits while zone 1 saw only 5.


In [ ]:
import numpy as np  # reuse numpy for the zone counts
import matplotlib.pyplot as plt  # reuse pyplot for the zone bars
np.random.seed(9)  # fix the seed for reproducibility
zones = np.array([12, 5, 7, 20])  # toy visits per pitch quarter (sums to 44)
print('visits per zone:', list(zones), '=', int(zones.sum()), 'total')  # the raw histogram
print('busiest zone:', int(np.argmax(zones)), 'with', int(zones.max()), 'visits')  # expect zone 3, 20 visits
print('quietest zone:', int(np.argmin(zones)), 'with', int(zones.min()), 'visits')  # expect zone 1, 5 visits
fig, ax = plt.subplots(figsize=(6, 3.5))  # make one panel for the zone bars
ax.bar(['zone 0', 'zone 1', 'zone 2', 'zone 3'], zones, color='orangered')  # one bar per quarter
ax.set_ylabel('tracked visits')  # label the count axis
ax.set_title('Heatmap in one dimension: zone 3 glows (20 of 44 visits)')  # title the takeaway
plt.tight_layout()  # snug the layout
plt.show()  # render the figure


## How it all fits together — the system map (§8)

One clip flows through every § term in order. Pixels enter **detection** (§4: color masks → blobs → centroid boxes, one set per frame). Boxes enter **tracking** (§5: GATE links, coasting bridges gaps, each survivor earning one ID). Track trails enter the **homography** (§6: DLT matrix turns pixel paths into meter paths with ~0 corner error). Meter paths exit as **stats** (§7: distances, top speeds, heatmap). The trace below walks a single centroid through the last two stations: frame-center pixel (80, 60) becomes (20.0, 15.0) m, and a short two-frame jog converts to meters of running.


In [ ]:
import numpy as np  # import numpy for the seed habit (diagram uses no randomness)
import matplotlib.pyplot as plt  # only pyplot draws: this cell is a pure diagram
np.random.seed(9)  # fix the seed for reproducibility
fig, ax = plt.subplots(figsize=(10, 3.5))  # wide figure for the left-to-right pipeline
ax.axis('off')  # hide axes: diagram, not data
ax.set_xlim(0, 14)  # fix placement range
ax.set_ylim(0, 4)  # fix placement range
stages = [('pixels\nframes', 1), ('detections §4\nboxes+centroids', 4), ('tracks §5\nIDs+coast', 7), ('H §6\npx -> m', 9.8), ('stats §7\ndist/heat', 12.3)]  # pipeline stations
colors = ['wheat', 'lightblue', 'plum', 'lightgreen', 'steelblue']  # color per station type
for (name, xc), c in zip(stages, colors):  # draw each pipeline station
    ax.text(xc, 2.2, name, ha='center', va='center', fontsize=8,  # centered station label
            bbox=dict(boxstyle='round', facecolor=c))  # rounded colored box around it
ax.text(6.8, 3.5, 'ANALYZE: pixels -> boxes (§4) -> trails (§5) -> meters (§6) -> numbers (§7)', ha='center', fontsize=9, fontweight='bold')  # banner line
for xa, xb in [(1.9, 3.0), (5.1, 6.1), (7.9, 9.1), (10.5, 11.6)]:  # arrow gaps between stations
    ax.annotate('', xy=(xb, 2.2), xytext=(xa, 2.2), arrowprops=dict(arrowstyle='->'))  # forward arrow left -> right
ax.set_title('System map: one clip from pixels to meters')  # diagram title
plt.tight_layout()  # snug the layout
plt.show()  # render the map


In [ ]:
import numpy as np  # reuse numpy for the DLT + mapping trace
np.random.seed(9)  # fix the seed (arrays below are hardcoded; habit kept)
SRC = np.array([[12., 12.], [148., 12.], [148., 108.], [12., 108.]])  # lab pitch-rect corners in px (§6)
DST = np.array([[0., 0.], [40., 0.], [40., 30.], [0., 30.]])  # same corners in meters (§6)
A = []  # DLT rows (§6)
for (x, y), (X, Y) in zip(SRC, DST):  # one corner pair at a time
    A.append([-x, -y, -1, 0, 0, 0, X * x, X * y, X])  # X-equation row
    A.append([0, 0, 0, -x, -y, -1, Y * x, Y * y, Y])  # Y-equation row
_, _, Vt = np.linalg.svd(np.array(A))  # null space holds the matrix
Hm = Vt[-1].reshape(3, 3) / Vt[-1].reshape(3, 3)[2, 2]  # reshape + normalize
def apply_H(Hm, pts):  # pixels -> meters (§6)
    p = np.asarray(pts, dtype=float)  # float array
    h = (Hm @ np.concatenate([p, np.ones((len(p), 1))], axis=1).T).T  # homogeneous multiply
    return h[:, :2] / h[:, 2:3]  # divide by w
pix = [(80., 60.), (90., 65.)]  # one centroid, then its next-frame jog (§4 positions)
m = apply_H(Hm, pix)  # both points in meters (§6)
step = float(np.linalg.norm(m[1] - m[0]))  # jog length in meters (§7 distance math)
print('centroid px', pix[0], '-> meters', np.round(m[0], 2))  # expect [20. 15.]: dead center
print('next-frame px', pix[1], '-> meters', np.round(m[1], 2))  # the jog endpoint
print('one-frame jog =', round(step, 3), 'm (x 10 fps = m/s)')  # distance-to-speed in one multiply


## Bridge to the lab

Open `notebook.ipynb` now. Its cells implement exactly the map above, on a synthetic match clip:

1. **Cell 1 (setup)** — seeds plus `os.makedirs('./results')`. Same `np.random.seed` habit as this lecture.
2. **Cell 2 (synthetic clip)** — simulates a 48-frame 160×120 clip (8 players + ball on a drawn pitch with 6% occlusion) — ground-truth positions are known, so recall and tracking mistakes score exactly.
3. **Cell 3 (detector from scratch)** — `label_components` (BFS blob grouping, your §4 flood-fill) plus color thresholds turn pixels into boxes; the lab reports recall 412/415 ≈ 0.993 at ~3.3 ms/frame — the "crayon circling" step.
4. **Cell 4 (greedy tracker)** — per-group nearest-centroid matching within GATE = 12 px with coasting up to 3 missed frames (§5) produces 11 IDs for 9 entities (one occlusion split = one ID switch).
5. **Cell 5 (homography to meters)** — `homography_dlt` solves the 3×3 matrix from the 4 pitch corners (§6, reprojection error ~1e-12 m) and exports `results/tracks.csv` plus per-player distance/top-speed in `results/summary.csv`.
6. **Cell 6 (figure)** — `results/pipeline.png` shows frame + trajectories in meters + heatmap (§7) — the analyst's final deliverable.


## Self-check — predict first, then check

> Retrieval rule: cover every *Answer* below, write your prediction (a number, a shape, or one sentence), run the § cell it cites, then uncover. The last question in each set has no printed answer — do it from the map.



1. **What are the three stages of the sports pipeline, and what does each output?**

*Answer: detect (boxes + centroids per frame, §4) → track (named trails across frames with coasting through gaps, §5) → homography (trails in meters, distances, speeds, heatmap, §6–§7). Point at each station of the §8 map as you narrate.*

2. **Mia is hidden for 2 frames with MAX_MISS = 3. What happens, and what if she hid for 5?**

*Answer: the track coasts at its last spot (misses 1, then 2) and re-links on return (§5's timeline) — but 5 straight misses exceeds patience 3, so the tag dies and her return births a new ID: an ID switch.*

3. **Why do 4 known pitch corners determine the homography, and how do you check it?**

*Answer: each corner pair contributes 2 equations pinning down the 8 unknowns of the normalized 3×3 matrix (DLT, §6) — then reprojection error re-maps the corners and reports the worst mismatch (lab: 2.0e-12 m, i.e. a perfect fit).*
